# TextClassification

## Importing required libraries

In [1]:
import pandas as pd

## Loading the Data
source: https://www.kaggle.com/ashishpatel26/sentimental-analysis-nlp

In [2]:
df = pd.read_csv('Datasets/sentiment_data.csv', header=None, names=['Label', 'Text'], sep='\t')

## Data Cleaning and Preparation

### Check the dimensions 

In [3]:
df.shape

(6918, 2)

### Check columns names

In [4]:
df.columns

Index(['Label', 'Text'], dtype='object')

In [5]:
df.sample(10)

,Label,Text
2016,1,I miss the Harry Potter Hookup.
4048,0,"to see the da vinci code, which totally sucked..."
6571,0,My dad's being stupid about brokeback mountain...
2500,1,I am going to start reading the Harry Potter s...
4651,0,"Da Vinci Code = Up, Up, Down, Down, Left, Righ..."
3490,1,I either LOVE Brokeback Mountain or think it's...
2983,1,I love the Harry Potter series if you can coun...
5031,0,And this is why I hate Harry Potter.
5243,0,"Not because I hate Harry Potter, but because I..."
3652,1,Brokeback mountain was beautiful...


### Check for Nulls

In [6]:
df.isnull().values.any()

False

## Machine Learning: LinearSVC

In [7]:
import sklearn
from sklearn.svm import LinearSVC
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import accuracy_score

### Define Output and Inputs

In [8]:
y = df['Label']
X = df['Text']

### Split dataset

In [9]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

In [10]:
X_train.shape, X_test.shape

((5534,), (1384,))

In [11]:
y_train.shape, y_test.shape

((5534,), (1384,))

### Vectorizer

In [12]:
tfidf_vect = TfidfVectorizer(max_features=15)

X_trans= tfidf_vect.fit_transform(X_train)

### Instantiate LinearSVC

In [13]:
classifier = LinearSVC(C=1.0, max_iter=1000, tol=1e-3)

linear_svc_model = classifier.fit(X_trans, y_train)

In [14]:
X_test_trans = tfidf_vect.fit_transform(X_test)

In [15]:
y_pred = linear_svc_model.predict(X_test_trans)

### Score

In [16]:
pred_results = pd.DataFrame({'y_test': y_test, 'y_pred': y_pred})

In [17]:
accuracy = accuracy_score(y_test, y_pred)
accuracy

0.9082369942196532

## Save model without preprocessor

### Create a Checkpoint

In [18]:
text_clf_param = {}

text_clf_param['preprocessing'] = tfidf_vect
text_clf_param['model'] = linear_svc_model
text_clf_param['sklearn_version'] = sklearn.__version__
text_clf_param['accuracy'] = accuracy

text_clf_param

{'preprocessing': TfidfVectorizer(max_features=15),
 'model': LinearSVC(tol=0.001),
 'sklearn_version': '0.24.2',
 'accuracy': 0.9082369942196532}

### Save

In [19]:
import joblib

In [20]:
filename = 'Models/text_clf_model_checkpoint.joblib'

In [21]:
joblib.dump(text_clf_param, filename)

['Models/text_clf_model_checkpoint.joblib']

### Load

In [22]:
clf_checkpoint = joblib.load(filename)

In [23]:
reloaded_vect = clf_checkpoint['preprocessing']

In [24]:
clf_model = clf_checkpoint['model']

### Retrain

In [25]:
X_test_trans_new = tfidf_vect.fit_transform(X_test)

y_pred = clf_model.predict(X_test_trans_new)

In [26]:
accuracy = accuracy_score(y_test, y_pred)
accuracy

0.9082369942196532

## Uses Pipeline
Allows to specifiy all of the preprocessing you want to perform on your data

In [27]:
from sklearn.pipeline import Pipeline

In [28]:
clf_pipeline = Pipeline(steps=[('tfidf_vect', tfidf_vect), ('classifier', classifier)])

pipeline_model = clf_pipeline.fit(X_train, y_train)

In [29]:
y_pred = pipeline_model.predict(X_test) #can use data directly

In [30]:
accuracy = accuracy_score(y_test, y_pred)
accuracy

0.9082369942196532

### Create a Checkpoint

In [31]:
pipe_clf_param = {}

pipe_clf_param['pipeline_clf'] = pipeline_model
pipe_clf_param['sklearn_version'] = sklearn.__version__
pipe_clf_param['accuracy'] = accuracy

pipe_clf_param

{'pipeline_clf': Pipeline(steps=[('tfidf_vect', TfidfVectorizer(max_features=15)),
                 ('classifier', LinearSVC(tol=0.001))]),
 'sklearn_version': '0.24.2',
 'accuracy': 0.9082369942196532}

### Save

In [32]:
filename = 'Models/pipe_clf_model_checkpoint.joblib'

In [33]:
joblib.dump(pipe_clf_param, filename)

['Models/pipe_clf_model_checkpoint.joblib']

### Load

In [34]:
pipe_clf_checkpoint = joblib.load(filename)

In [35]:
reloaded_pipeline = pipe_clf_checkpoint['pipeline_clf']

### Test

In [36]:
y_pred = reloaded_pipeline.predict(X_test)

In [38]:
accuracy_score(y_test, y_pred)

0.9082369942196532